# Save-As-You-Go: Streaming DAQ Data to HDF5

This notebook demonstrates versionable's incremental HDF5 persistence.
With HDF5 Sessions, data is written to disk as it arrives — no need to hold everything in memory.

> **Requirements:** `plotly` and `jupyterlab`

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path

import numpy as np
import numpy.typing as npt
import plotly.graph_objects as go
from daq_simulator import DaqSimulator

import versionable
import versionable.hdf5  # Will error if h5py isn't installed
from versionable import Versionable

## 1. Define the Recording type

A normal Versionable dataclass. Inside a session, all ndarray fields are
automatically backed by resizable HDF5 datasets — no annotation needed.

In [ ]:
@dataclass
class Recording(Versionable, version=1, hash="74a182"):
    """A single DAQ recording with time-series data."""

    name: str = ""
    sampleRate_Hz: float = 0.0
    startTime: datetime = field(default_factory=lambda: datetime.now().astimezone())
    time_s: npt.NDArray[np.float64] = field(default_factory=lambda: np.empty(0))
    voltage_V: npt.NDArray[np.float64] = field(default_factory=lambda: np.empty(0))

## 2. Record the first session

`open()` accepts either a **class** (empty proxy) or an **existing instance**.
Here we pass an instance — only set what you care about, defaults handle the rest.

> The `DaqSimulator` is defined in [`daq_simulator.py`](daq_simulator.py) — a simple
> generator that yields chunks of a sinusoidal signal.

In [ ]:
OUTPUT_PATH = Path("recording.h5")

daq = DaqSimulator(frequency_Hz=10.0, amplitude_V=1.0, sampleRate_Hz=1000.0, duration_s=1.0)

recOriginal = Recording(name="10 Hz sinusoid", sampleRate_Hz=daq.sampleRate_Hz)

with versionable.hdf5.open(recOriginal, OUTPUT_PATH, mode="overwrite") as recOriginal:
    for tChunk, vChunk in daq.stream():
        recOriginal.time_s.append(tChunk)
        recOriginal.voltage_V.append(vChunk)

print(f"Wrote {OUTPUT_PATH} ({OUTPUT_PATH.stat().st_size / 1024:.1f} KB)")

## 3. Load and plot the first session

The file is a standard HDF5 — load it with `versionable.load()`, no special API.

In [ ]:
# Load the recording from disk.
firstRec = versionable.load(Recording, OUTPUT_PATH)

# Print some info and plot the data.
print(f"{firstRec.name} — {len(firstRec.time_s)} samples, {firstRec.sampleRate_Hz:.0f} Hz")
print(f"Duration: {firstRec.time_s[-1] - firstRec.time_s[0]:.3f} s")
fig = go.Figure(layout={"title": "First session (1.0 s)", "xaxis_title": "Time (s)", "yaxis_title": "V"})
fig.add_trace(go.Scatter(x=firstRec.time_s, y=firstRec.voltage_V))
fig.show()

## 4. Resume and append more data

Reopen with `mode="resume"` — existing data is restored, and new appends
continue from where we left off.

> **NOTE**: Running this multiple times will keep appending the data

In [ ]:
# Open the same file in "resume" mode, which allows appending to existing datasets.
# The previous data is also accessible through the session object.
with versionable.hdf5.open(Recording, OUTPUT_PATH, mode="resume") as rec2:
    # Figure out where the previous session left off do the simualtor can resume from there.
    startOffset = float(rec2.time_s[-1]) + 1.0 / rec2.sampleRate_Hz
    print(f"Resuming from {len(rec2.time_s)} samples (t={startOffset:.4f} s)")

    # Append another 0.5 seconds
    daq2 = DaqSimulator(frequency_Hz=10.0, amplitude_V=1.0, sampleRate_Hz=1000.0, duration_s=0.5)

    for tChunk, vChunk in daq2.stream(startOffset_s=startOffset):
        rec2.time_s.append(tChunk)
        rec2.voltage_V.append(vChunk)

    print(f"Total samples: {len(rec2.time_s)}")

## 5. Load and plot the combined result

The file now contains all captures. (Try running the cell above a few times).

In [ ]:
combined = versionable.load(Recording, OUTPUT_PATH)
combinedTime_s = combined.time_s[-1] - combined.time_s[0]
print(f"Total: {len(combined.time_s)} samples, {combinedTime_s:.1f} s")

fig = go.Figure(layout={"title": f"Combined ({combinedTime_s:.1f} s)", "xaxis_title": "Time (s)", "yaxis_title": "V"})
fig.add_trace(go.Scatter(x=combined.time_s, y=combined.voltage_V))
fig.show()

## Cleanup

In [ ]:
OUTPUT_PATH.unlink(missing_ok=True)  # Delete the file when done
print(f"Deleted {OUTPUT_PATH.name}")